In [1]:
!pip install flask


   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   ---------------------------------------- 0/5 [werkzeug]
   -------- ------------------------------- 1/5 [itsdangerous]
   ---------------- ----------------------- 2/5 [click]
   ---------------- ----------------------- 2/5 [click]
   ---------------- ----------------------- 2/5 [click]
   ------------------------ --------------- 3/5 [blinker]
   -------------------------------- ------- 4/5 [flask]
   -------------------------------- ------- 4/5 [flask]
   -------------------------------- ------- 4/5 [flask]
   -------------------------------- ------- 4/5 [flask]
   ---------------------------------------- 5/5 [flask]



In [2]:
from flask import Flask, render_template_string, request
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

app = Flask(__name__)

# 모델 준비
model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

chat_history = ""

@app.route("/", methods=["GET", "POST"])
def index():
    global chat_history
    bot_reply = ""
    if request.method == "POST":
        user_input = request.form["message"]
        chat_history += f"User: {user_input}\nBot:"
        inputs = tokenizer(chat_history, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            max_length=200,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_p=0.9,
            temperature=0.8
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        bot_reply = response.split("Bot:")[-1].split("User:")[0].strip()
        chat_history += f" {bot_reply}\n"

    return render_template_string("""
        <h2>한국어 GPT2 챗봇</h2>
        <form method="post">
            <input name="message" placeholder="메시지를 입력하세요" style="width:300px">
            <input type="submit" value="Send">
        </form>
        <p><b>Bot:</b> {{bot_reply}}</p>
    """, bot_reply=bot_reply)

if __name__ == "__main__":
    app.run(debug=True)


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\IPython\core\interactiveshell.py:3680: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
